# Task 3: OAS2 target checks on chromosome 12

This notebook adds the missing chromosome 12 target check for **200forw**
and **248forw**. It does not change the chromosome 1/2 off-target results.

1. Put a **GRCh38 chromosome 12 VCF** beside this notebook, or change `VCF_PATH` below.
   Use the same seven individuals and, where possible, the same dataset/release as before.
   Plain `.vcf` and compressed `.vcf.gz` files are supported. No index is required.
2. Select a Python kernel in VS Code or Jupyter and run the cells in order.
3. If the VCF header does not identify its assembly, check the dataset documentation,
   then set `VCF_ASSEMBLY = "GRCh38"`. Do not use GRCh37 coordinates with these targets.
4. Keep the files written to `chr12_results`. They record the results, reference,
   input header and actual software versions.

The notebook uses Python's standard library and **pandas**. If pandas is missing,
run `%pip install pandas` in a separate cell, then restart the kernel.
Internet access is needed for the first reference-sequence check. Later runs reuse
the saved reference FASTA. The VCF itself is not downloaded by this notebook.

**Scope:** SNP alleles are reconstructed for both copies. Indels, other complex
variants, missing genotypes and uncertain local phase are flagged for review.
Absence of a variant record does not establish reliable sequence coverage.

In [9]:
from pathlib import Path
from urllib.request import Request, urlopen
from datetime import datetime, timezone
import gzip, json, platform, re, sys, shutil, tempfile, zipfile
import pandas as pd

VCF_PATH = Path(
    r"C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3"
    r"\ALL.chr12.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf"
    r"\ALL.chr12.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf"
)

if not VCF_PATH.is_file():
    raise FileNotFoundError(
        f"Die VCF-Datei wurde unter diesem Pfad nicht gefunden:\n{VCF_PATH}"
    )

# Prüfen, ob die Datei entpackt ist und einen VCF-Header enthält.
with VCF_PATH.open("rb") as stream:
    anfang = stream.read(100)

if anfang.startswith(b"\x1f\x8b"):
    raise ValueError(
        "Die Datei ist noch gzip-komprimiert. "
        "Verwende die vollständig entpackte VCF-Datei."
    )

if not anfang.startswith(b"##fileformat=VCF"):
    raise ValueError(
        "Die Datei beginnt nicht mit einem gültigen VCF-Header."
    )

gzip_input = False
VCF_IN_ZIP = None
input_archive = None
input_zip_member = None

VCF_ASSEMBLY = "GRCh38"
VCF_SOURCE = "IGSR 20190312_biallelic_SNV_and_INDEL, chromosome 12"

OUTPUT_DIR = Path("chr12_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = [
    "HG00142",
    "HG00178",
    "HG00237",
    "HG01521",
    "HG03129",
    "NA19175",
    "NA19462",
]

TARGETS = {
    "200forw": {
        "start": 112978698,
        "end": 112978720,
        "reference": "TCAGACACTGATCGACGAGATGG",
    },
    "248forw": {
        "start": 112978746,
        "end": 112978768,
        "reference": "GGAACCCGAACAGTTCCCCCTGG",
    },
}

print("Gefundene Datei:", VCF_PATH.resolve())
print("Dateiformat: entpackte VCF")
print("Ergebnisordner:", OUTPUT_DIR.resolve())
print("Python:", sys.version)
print("pandas:", pd.__version__)

Gefundene Datei: C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3\ALL.chr12.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf\ALL.chr12.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf
Dateiformat: entpackte VCF
Ergebnisordner: C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3\chr12_results
Python: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
pandas: 2.3.3


## Check the GRCh38 reference sequence

The coordinates above were calculated from the first-exon sequence in Task 2.
This cell checks them against Ensembl's GRCh38 reference and saves that sequence.
It stops if either expected 23-base sequence does not match. This verifies the
reference targets; the VCF assembly is checked separately in the next cell.

In [14]:
EXON_START, EXON_END = 112978519, 112978785
REFERENCE_URL = (
    "https://rest.ensembl.org/sequence/region/human/"
    f"12:{EXON_START}..{EXON_END}:1?coord_system_version=GRCh38;content-type=text/plain"
)
reference_file = OUTPUT_DIR / "Task3_chr12_reference_exon.fasta"
if reference_file.exists():
    exon_sequence = "".join(line.strip() for line in reference_file.read_text().splitlines()
                            if not line.startswith(">")).upper()
else:
    request = Request(REFERENCE_URL, headers={"Accept": "text/plain"})
    with urlopen(request, timeout=30) as response:
        exon_sequence = "".join(response.read().decode().split()).upper()
if len(exon_sequence) != EXON_END - EXON_START + 1 or re.fullmatch("[ACGT]+", exon_sequence) is None:
    raise ValueError("Unexpected reference response. Check the Ensembl request or cached FASTA.")
for guide, target in TARGETS.items():
    observed = exon_sequence[target["start"] - EXON_START:target["end"] - EXON_START + 1]
    if observed != target["reference"]:
        raise ValueError(f"Reference mismatch for {guide}. Check Task 2 coordinates before continuing.")
    print(guide, f'chr12:{target["start"]}-{target["end"]}', observed)
reference_file.write_text(f">GRCh38_chr12_{EXON_START}_{EXON_END}_forward\n{exon_sequence}\n")

200forw chr12:112978698-112978720 TCAGACACTGATCGACGAGATGG
248forw chr12:112978746-112978768 GGAACCCGAACAGTTCCCCCTGG


310

## Read chromosome 12 and retain overlapping records

The file is read one line at a time, so a full chromosome table is not held in memory.
`12` and `chr12` are both accepted. The REF span and `INFO/END`, when present,
are considered so that a deletion starting before a target can be retained.
This remains a local target check, not a genome-wide structural-variant analysis.

In [11]:
records = {guide: [] for guide in TARGETS}
metadata, header, seen_chromosomes = [], None, set()
source_columns, assembly_evidence = None, None
opener = gzip.open if VCF_PATH.suffix.lower() == ".gz" else open
with opener(VCF_PATH, "rt", encoding="utf-8") as stream:
    for line_number, line in enumerate(stream, 1):
        if line.startswith("##"):
            metadata.append(line.rstrip())
            continue
        if line.startswith("#CHROM"):
            header = line.rstrip().split("\t")
            missing = set(SAMPLES) - set(header[9:])
            if missing:
                raise ValueError(f"Samples missing from the VCF: {sorted(missing)}")
            source_columns = {sample: header.index(sample) for sample in SAMPLES}
            text = "\n".join(m for m in metadata if m.startswith(("##reference=", "##assembly="))
                             or (m.startswith("##contig=") and "assembly=" in m))
            if re.search(r"GRCh37|hg19|hs37|g1k_v37", text, re.I):
                raise ValueError("The VCF metadata mentions GRCh37/hg19. Obtain matching GRCh38 data.")
            detected = "GRCh38" if re.search(r"GRCh38|hg38|hs38|assembly38", text, re.I) else None
            if (VCF_ASSEMBLY or detected) != "GRCh38":
                raise ValueError("Assembly is not identified as GRCh38. Check the VCF source, then set VCF_ASSEMBLY.")
            assembly_evidence = "VCF header" if detected else "User-confirmed dataset documentation"
            continue
        if line.startswith("#") or not line.strip():
            continue
        if source_columns is None:
            raise ValueError("VCF sample header is missing.")
        fields = line.rstrip().split("\t")
        seen_chromosomes.add(fields[0])
        if fields[0] not in ("12", "chr12"):
            continue
        position, ref = int(fields[1]), fields[3].upper()
        end_match = re.search(r"(?:^|;)END=(\d+)(?:;|$)", fields[7])
        end = max(position + len(ref) - 1, int(end_match.group(1)) if end_match else position)
        overlapping = [g for g, t in TARGETS.items() if position <= t["end"] and end >= t["start"]]
        if not overlapping:
            continue
        keys = fields[8].split(":")
        calls = {sample: dict(zip(keys, fields[index].split(":")))
                 for sample, index in source_columns.items()}
        record = {"POS": position, "END": end, "REF": ref,
                  "ALT": fields[4].upper().split(","), "FILTER": fields[6], "calls": calls}
        for guide in overlapping:
            target = TARGETS[guide]
            left, right = max(position, target["start"]), min(position + len(ref) - 1, target["end"])
            if left <= right and re.fullmatch("[ACGT]+", ref):
                reference_part = target["reference"][left-target["start"]:right-target["start"]+1]
                if reference_part != ref[left-position:right-position+1]:
                    raise ValueError(f"VCF REF mismatch at {position} for {guide}; check assembly/coordinates.")
            records[guide].append(record)
if header is None:
    raise ValueError("VCF sample header is missing.")
if seen_chromosomes and not seen_chromosomes.intersection({"12", "chr12"}):
    raise ValueError("The input contains no chromosome 12 records. Check the input file.")
print("GRCh38 basis:", assembly_evidence)
for guide, values in records.items():
    print(guide, ":", len(values), "overlapping VCF records")

GRCh38 basis: VCF header
200forw : 0 overlapping VCF records
248forw : 4 overlapping VCF records


## Compare both copies for each individual

A matching copy means an unchanged **20-base guide match plus a canonical NGG PAM**.
The first PAM base can vary. A protospacer mismatch does not prove loss of CRISPRi
activity; no efficiency or activity score is calculated here.

Copy numbers are local sequence reconstructions from the supplied calls. A single
unphased heterozygous SNP gives two distinguishable copies, but their order is arbitrary.
Multiple heterozygous sites require compatible phase information. VCF phased calls
without a PS field use the VCF convention of one implicit phase set.

In [12]:
def assess_target(guide, sample):
    target, values = TARGETS[guide], records[guide]
    copies = [list(target["reference"]), list(target["reference"])]
    issues, heterozygous, descriptions, alt_records = [], [], [], 0
    for record in values:
        call = record["calls"][sample]
        gt = call.get("GT", ".")
        descriptions.append(f'{record["POS"]}:{gt}')
        alleles = re.split(r"[|/]", gt.lstrip("|/"))
        if len(alleles) != 2 or any(re.fullmatch(r"\d+", a) is None for a in alleles):
            issues.append("Missing or unsupported genotype")
            continue
        alleles = [int(a) for a in alleles]
        if max(alleles) > len(record["ALT"]):
            issues.append("Invalid ALT allele index")
            continue
        if record["FILTER"] not in ("PASS", ".") or call.get("FT", ".") not in ("PASS", "."):
            issues.append("Filtered call requires review")
        if any(alleles):
            alt_records += 1
        if alleles[0] != alleles[1]:
            heterozygous.append(("|" in gt and "/" not in gt, call.get("PS", ".")))
            if call.get("PSL", ".") != ".":
                issues.append("PSL phase information requires review")
        for copy_number, allele_index in enumerate(alleles):
            if allele_index == 0:
                continue
            alternate = record["ALT"][allele_index - 1]
            if len(record["REF"]) != 1 or re.fullmatch("[ACGT]", alternate) is None:
                issues.append("Indel or complex variant requires review")
                continue
            offset = record["POS"] - target["start"]
            if not 0 <= offset < 23:
                issues.append("Complex overlap requires review")
                continue
            if copies[copy_number][offset] not in (target["reference"][offset], alternate):
                issues.append("Conflicting overlapping calls")
            copies[copy_number][offset] = alternate
    if len(heterozygous) > 1:
        if not all(phased for phased, ps in heterozygous) or len({ps for phased, ps in heterozygous}) > 1:
            issues.append("Uncertain phase across heterozygous sites")
    row = {"Guide": guide, "Sample": sample, "Start": target["start"], "End": target["end"],
           "Overlapping_records": len(values), "ALT_records": alt_records,
           "Genotypes": "; ".join(descriptions) or "No overlapping record",
           "Copy_1_sequence": None, "Copy_2_sequence": None,
           "Mismatches_1": None, "Mismatches_2": None, "PAM_1": None, "PAM_2": None,
           "Matching_copies_in_called_data": None,
           "Phase_basis": "VCF phase" if len(heterozygous) > 1 else "Copy order not needed for count",
           "Coverage_checked": False}
    if issues:
        row["Status"] = "; ".join(sorted(set(issues)))
        return row
    sequences = ["".join(copy) for copy in copies]
    matches = 0
    for n, sequence in enumerate(sequences, 1):
        mismatches = sum(a != b for a, b in zip(target["reference"][:20], sequence[:20]))
        row.update({f"Copy_{n}_sequence": sequence, f"Mismatches_{n}": mismatches,
                    f"PAM_{n}": sequence[20:]})
        matches += int(mismatches == 0 and sequence[21:] == "GG")
    row["Matching_copies_in_called_data"] = matches
    row["Status"] = ("No sequence change in supplied calls" if sequences == [target["reference"]] * 2
                     else "Sequence change in supplied calls")
    return row

summary = pd.DataFrame([assess_target(guide, sample) for guide in TARGETS for sample in SAMPLES])
for column in ["Mismatches_1", "Mismatches_2", "Matching_copies_in_called_data"]:
    summary[column] = summary[column].astype("Int64")
display(summary[["Guide", "Sample", "Mismatches_1", "Mismatches_2", "PAM_1", "PAM_2",
                 "Matching_copies_in_called_data", "Status"]])
print("Counts describe supplied variant calls. Coverage has not been checked.")

,Guide,Sample,Mismatches_1,Mismatches_2,PAM_1,PAM_2,Matching_copies_in_called_data,Status
0,200forw,HG00142,0,0,TGG,TGG,2,No sequence change in supplied calls
1,200forw,HG00178,0,0,TGG,TGG,2,No sequence change in supplied calls
2,200forw,HG00237,0,0,TGG,TGG,2,No sequence change in supplied calls
3,200forw,HG01521,0,0,TGG,TGG,2,No sequence change in supplied calls
4,200forw,HG03129,0,0,TGG,TGG,2,No sequence change in supplied calls
5,200forw,NA19175,0,0,TGG,TGG,2,No sequence change in supplied calls
6,200forw,NA19462,0,0,TGG,TGG,2,No sequence change in supplied calls
7,248forw,HG00142,0,0,TGG,TGG,2,No sequence change in supplied calls
8,248forw,HG00178,0,0,TGG,TGG,2,No sequence change in supplied calls
9,248forw,HG00237,0,0,TGG,TGG,2,No sequence change in supplied calls


Counts describe supplied variant calls. Coverage has not been checked.


## Save the results and actual environment

Blank/NA sequence results indicate that a call needs review; they are not zero matches.
Even a count of two is conditional on the supplied variant data and its coverage.
If no variant is detected, report: **"No overlapping variant was detected in the
analysed data."** Do not report confirmed guide availability from a variant-only
file alone. Retain the VCF source and any callable-region information.

The exports can be used to add a small chromosome 12 results table to Task 3.
Open this notebook's tables and check any flagged rows before changing the report.

In [13]:
variant_rows = []
for guide, values in records.items():
    for record in values:
        for sample in SAMPLES:
            call = record["calls"][sample]
            variant_rows.append({"Guide": guide, "Chromosome": "chr12", "POS": record["POS"],
                                 "END": record["END"], "REF": record["REF"],
                                 "ALT": ",".join(record["ALT"]), "FILTER": record["FILTER"],
                                 "Sample": sample, "GT": call.get("GT", "."),
                                 "PS": call.get("PS", "."), "GQ": call.get("GQ", "."),
                                 "DP": call.get("DP", ".")})
variants = pd.DataFrame(variant_rows, columns=["Guide", "Chromosome", "POS", "END", "REF", "ALT",
                                             "FILTER", "Sample", "GT", "PS", "GQ", "DP"])
summary.to_csv(OUTPUT_DIR / "Task3_chr12_on_target_summary.csv", index=False, na_rep="NA")
variants.to_csv(OUTPUT_DIR / "Task3_chr12_on_target_variants.csv", index=False)
(OUTPUT_DIR / "Task3_chr12_input_header.txt").write_text("\n".join(metadata + ["\t".join(header)]) + "\n")
environment = {"run_time_utc": datetime.now(timezone.utc).isoformat(), "Python": sys.version,
               "pandas": pd.__version__, "OS": platform.platform(), "input": str(VCF_PATH.resolve()),
               "input_size_bytes": VCF_PATH.stat().st_size, "VCF_source": VCF_SOURCE or "Not recorded",
               "assembly": "GRCh38", "assembly_evidence": assembly_evidence,
               "reference_url": REFERENCE_URL, "samples": SAMPLES, "targets": TARGETS,
               "overlap_records": {g: len(v) for g, v in records.items()},
               "coverage_checked": False, "SNP_reconstruction_only": True}
(OUTPUT_DIR / "Task3_chr12_environment.json").write_text(json.dumps(environment, indent=2))
print("Saved files:")
for path in sorted(OUTPUT_DIR.glob("Task3_chr12_*")):
    print(path.resolve())

Saved files:
C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3\chr12_results\Task3_chr12_environment.json
C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3\chr12_results\Task3_chr12_input_header.txt
C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3\chr12_results\Task3_chr12_on_target_summary.csv
C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3\chr12_results\Task3_chr12_on_target_variants.csv
C:\Users\piets\OneDrive\QUT\Biomedical Data Science\Project\Task 3\chr12_results\Task3_chr12_reference_exon.fasta
